[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C00_VLM_Multimodal_Course/09_native_multimodal_agents/09_native_agents.ipynb)

# 09 · 原生多模态与多模态智能体 — GUI grounding & 感知-行动循环

> 配套模块 09 讲解。本 notebook 用真实可运行代码演示**多模态智能体**最核心的两块基础能力：
> 1. **GUI grounding** —— 把"点击某按钮"这类指令对应到屏幕上的坐标 / bounding box，并可视化；
> 2. **Set-of-Marks (SoM) 提示** —— 在界面上叠加编号标记，把坐标回归降级为"选编号"，提升 grounding 可靠性；
> 3. **最简感知-行动循环 (perception-action loop)** —— 在一个 **toy 环境（纯模拟，绝不触碰真实 OS）** 里跑通 `观察→推理→动作→执行` 的 agent 骨架；
> 4. **(可选) 调用闭源原生多模态** —— 若检测到 `OPENAI_API_KEY`，对一张图调用 GPT-4o 做一次多模态问答。
> 5. **3 道 ✏️ 练习** —— bbox IoU 与点击判定、0–1000 归一化坐标互转、鲁棒 JSON action 解析，全部 CPU 纯 Python 可跑。
>
> **算力**：grounding 部分用 `Qwen/Qwen2-VL-2B-Instruct`（具 grounding 能力），fp16 下约需 **5–6 GB 显存**，Colab T4（16GB）够用、不需要量化；不做训练、不下载大模型权重以外的东西。
>
> ⚠️ **安全声明**：本 notebook 的"动作"全部作用在一个 Python 字典表示的**模拟界面**上，纯教学演示，**不调用任何真实鼠标/键盘/系统 API**。生产级 computer-use 必须配沙箱、动作白名单与人类确认。

## 1. 环境与设备

统一的 import + 设备探测 + 版本打印。GUI grounding 推荐 GPU；CPU 也能跑但很慢。

In [ ]:
import sys, torch
from PIL import Image, ImageDraw, ImageFont
import transformers

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("python       :", sys.version.split()[0])
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("device       :", device)

# Qwen2-VL 需要较新的 transformers（>= 4.45）。若报 ImportError，请升级。
# pip install -U "transformers>=4.45" accelerate qwen-vl-utils
# 2B 模型 fp16 只要 ~5-6GB，T4(16GB) 默认不需要 bitsandbytes 量化。


## 2. 准备一张界面截图

我们优先尝试下载一张真实的 UI 截图；如果无网络/下载失败，就用 PIL **现画一个含若干按钮的简单界面**作为兜底。
两种情况下后续代码完全一致——这保证 notebook 在离线环境也能跑通。

In [ ]:
import io, urllib.request

W, H = 640, 400  # 画布尺寸（兜底界面用）

def draw_toy_ui():
    # 用 PIL 画一个简单的 toy 界面：标题栏 + 几个按钮 + 一个搜索框。
    img = Image.new("RGB", (W, H), (245, 247, 250))
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 18)
        small = ImageFont.truetype("DejaVuSans.ttf", 14)
    except Exception:
        font = ImageFont.load_default(); small = font
    # 标题栏
    d.rectangle([0, 0, W, 48], fill=(60, 90, 160))
    d.text((16, 14), "Demo App", fill="white", font=font)
    # 顶部菜单
    for i, label in enumerate(["File", "Edit", "View"]):
        d.text((220 + i * 70, 14), label, fill="white", font=small)
    # 搜索框
    d.rectangle([40, 90, 400, 130], outline=(120, 120, 120), width=2, fill="white")
    d.text((52, 100), "Search...", fill=(150, 150, 150), font=small)
    # 三个按钮：Cancel / Save / Submit
    buttons = [("Cancel", (40, 300), (160, 130, 130)),
               ("Save",   (250, 300), (130, 160, 130)),
               ("Submit", (460, 300), (90, 140, 220))]
    for label, (x, y), color in buttons:
        d.rounded_rectangle([x, y, x + 140, y + 50], radius=8, fill=color)
        d.text((x + 40, y + 14), label, fill="white", font=font)
    return img

ui_img = None
URL = "https://raw.githubusercontent.com/githubnext/monaspace/main/README.md"  # 占位；真实截图源不稳定时直接走兜底
try:
    # 真实 UI 截图源经常变动/限流，这里默认直接走 PIL 兜底以保证可复现。
    raise RuntimeError("skip-download-use-local")
except Exception as e:
    print("未下载在线截图（", e, "），使用 PIL 自绘界面。")
    ui_img = draw_toy_ui()

print("界面尺寸:", ui_img.size)
ui_img


## 3. 加载 Qwen2-VL-2B（fp16）做 GUI grounding

`Qwen/Qwen2-VL-2B-Instruct` 具备 grounding / 定位能力。默认用 fp16 加载（~5–6 GB，Colab T4 够用，不需要量化）；
下面留了一个 `USE_4BIT` 开关给显存特别紧张的场景，默认关闭。

**API 要点**（当前 `transformers` 正确用法）：
- 类：`Qwen2VLForConditionalGeneration` + `AutoProcessor`
- 多模态对话用 `messages`（含 image + text）→ `processor.apply_chat_template(...)` → `process_vision_info` 取图 → `processor(...)` 组 batch
- **坐标约定**：Qwen2-VL 输出的 bbox 坐标是相对于**模型实际看到的（resize 后）图像**的绝对像素；processor 会把图缩放到某个尺寸，因此我们读回 `image_grid_thw` / 输入图尺寸来把坐标映射回原图。下面解析时做了稳健处理（兼容 0–1000 归一化与绝对像素两种返回）。

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_4BIT = False   # 2B 模型 fp16 只要 ~5-6GB，T4(16GB) 默认不需要量化；显存特别紧张时可自己改成 True

load_kwargs = {}
if device == "cuda":
    load_kwargs.update(dict(device_map="auto"))
    if USE_4BIT:
        from transformers import BitsAndBytesConfig
        load_kwargs.update(dict(
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16,
            ),
        ))
    else:
        load_kwargs.update(dict(torch_dtype=torch.float16))
else:
    # CPU/MPS 上用 fp32，速度较慢。
    load_kwargs.update(dict(torch_dtype=torch.float32))

print("加载", MODEL_ID, f"（fp16={'quantization_config' not in load_kwargs}，首次会下载约 4GB 权重）...")
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
processor = AutoProcessor.from_pretrained(MODEL_ID)
if device != "cuda":
    model = model.to(device)
model.eval()
print("加载完成。")


## 4. 下达 grounding 指令并解析坐标

我们让模型定位 **Submit 按钮**。Qwen2-VL 习惯用特殊标记输出框，例如：
`<|object_ref_start|>Submit<|object_ref_end|><|box_start|>(x1,y1),(x2,y2)<|box_end|>`。

下面用正则把括号里的数字抠出来，并做**稳健映射**：若数值最大值 ≤ 1000 视为千分比归一化坐标，按图像尺寸还原；否则视为绝对像素。最后用 PIL 在原图上画框。

In [ ]:
import re
from qwen_vl_utils import process_vision_info

def ask_qwen(image, prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": prompt},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
    return processor.batch_decode(trimmed, skip_special_tokens=False)[0]

prompt = "Locate the Submit button and output its bounding box."
raw = ask_qwen(ui_img, prompt)
print("模型原始输出：\n", raw)


In [ ]:
def parse_boxes(text, img_w, img_h):
    # 从模型输出里抠出所有 (x,y) 数对，配对成 bbox，并稳健映射回原图像素。
    nums = [float(n) for n in re.findall(r'-?\d+\.?\d*', text)]
    if len(nums) < 4:
        return []
    # 取前 4 个数当作 x1,y1,x2,y2
    x1, y1, x2, y2 = nums[:4]
    # 稳健判定：若坐标都 <= 1000 且图像本身更大，按千分比归一化还原
    if max(x1, y1, x2, y2) <= 1000 and max(img_w, img_h) > 1000 * 0.5 and max(x1,y1,x2,y2) <= 1000:
        # Qwen2-VL 常用 0-1000 归一化；映射回原图
        x1, x2 = x1 / 1000 * img_w, x2 / 1000 * img_w
        y1, y2 = y1 / 1000 * img_h, y2 / 1000 * img_h
    # clip 到画布内
    x1, x2 = sorted([max(0, min(img_w, x1)), max(0, min(img_w, x2))])
    y1, y2 = sorted([max(0, min(img_h, y1)), max(0, min(img_h, y2))])
    return [(x1, y1, x2, y2)]

boxes = parse_boxes(raw, *ui_img.size)
print("解析出的 bbox（原图像素）:", boxes)

vis = ui_img.copy()
d = ImageDraw.Draw(vis)
for (x1, y1, x2, y2) in boxes:
    d.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=4)
    d.text((x1, max(0, y1 - 16)), "Submit?", fill=(255, 0, 0))
print("红框 = 模型定位结果。若偏移，多因 2B 小模型 + 自绘界面分布外所致——这正是 grounding 的难点。")
vis


## 5. Set-of-Marks (SoM) 提示：把坐标回归降级为"选编号"

直接回归像素坐标对 2B 小模型偏难。**Set-of-Marks** 的思路：先（用已知布局 / 检测器）在界面上叠加带编号的标记，
再问模型"该点哪个编号"。这样模型只需做**离散选择**，而非连续坐标回归——可靠性显著提升，且把**感知（定位）与决策（选谁）解耦**。

这里我们用界面的已知控件布局直接叠标记（真实系统中编号来自检测/可访问性树 accessibility tree）。

In [ ]:
# 已知控件布局（真实系统里来自 a11y tree / 检测器；这里直接用我们画图时的坐标中心）
MARKS = {
    1: ("File menu",   (235, 22)),
    2: ("Edit menu",   (305, 22)),
    3: ("View menu",   (375, 22)),
    4: ("Search box",  (220, 110)),
    5: ("Cancel btn",  (110, 325)),
    6: ("Save btn",    (320, 325)),
    7: ("Submit btn",  (530, 325)),
}

def overlay_marks(image, marks):
    img = image.copy()
    d = ImageDraw.Draw(img)
    try:
        f = ImageFont.truetype("DejaVuSans.ttf", 16)
    except Exception:
        f = ImageFont.load_default()
    for idx, (_, (cx, cy)) in marks.items():
        r = 13
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(255, 80, 0), outline="white", width=2)
        d.text((cx - 4, cy - 9), str(idx), fill="white", font=f)
    return img

som_img = overlay_marks(ui_img, MARKS)

som_prompt = (
    "The screenshot has numbered markers on interactive elements. "
    "To submit the form, which numbered marker should be clicked? "
    "Answer with ONLY the number."
)
som_raw = ask_qwen(som_img, som_prompt, max_new_tokens=16)
print("SoM 模型输出：", repr(som_raw))
m = re.search(r'\d+', som_raw)
chosen = int(m.group()) if m else None
print("模型选择的编号:", chosen, "→", MARKS.get(chosen, ("(超出范围)", None))[0])
som_img


## 6. 最简感知-行动循环（toy 环境，纯模拟）

现在把"看屏→决策→动作→执行"连成闭环。**环境是一个 Python 字典**，渲染成图给 VLM；VLM 输出动作（点哪个编号 / 输入文本）；
我们在字典上 `apply` 这个动作，进入下一轮。**全程不触碰真实 OS**。

任务：*在搜索框输入 "cats"，然后提交*。我们跑 2–3 步演示骨架。

In [ ]:
# ---- toy 环境：一个表单 ----
class ToyFormEnv:
    def __init__(self):
        self.state = {"search_text": "", "submitted": False}
        self.action_log = []

    def render(self):
        # 把状态渲染成一张带编号标记的界面图（复用上面的画法）。
        img = Image.new("RGB", (W, H), (245, 247, 250))
        d = ImageDraw.Draw(img)
        try: f = ImageFont.truetype("DejaVuSans.ttf", 16)
        except Exception: f = ImageFont.load_default()
        d.rectangle([0, 0, W, 48], fill=(60, 90, 160))
        d.text((16, 14), "Demo App", fill="white", font=f)
        # 搜索框（显示当前文本）
        d.rectangle([40, 90, 400, 130], outline=(120, 120, 120), width=2, fill="white")
        shown = self.state["search_text"] or "Search..."
        d.text((52, 100), shown, fill=(20, 20, 20) if self.state["search_text"] else (150,150,150), font=f)
        # Submit 按钮
        col = (90, 140, 220) if not self.state["submitted"] else (90, 200, 120)
        d.rounded_rectangle([460, 300, 600, 350], radius=8, fill=col)
        d.text((500, 314), "Submit" if not self.state["submitted"] else "Done!", fill="white", font=f)
        return overlay_marks(img, {4: ("Search box", (220, 110)), 7: ("Submit btn", (530, 325))})

    def apply(self, action):
        self.action_log.append(action)
        kind = action.get("type")
        if kind == "type":
            self.state["search_text"] = action.get("text", "")
        elif kind == "click" and action.get("mark") == 7:
            if self.state["search_text"]:
                self.state["submitted"] = True
        return self.state

env = ToyFormEnv()
env.render()


In [ ]:
import json as _json

# ---- 简单的动作解析：让 VLM 以 JSON 输出动作 ----
ACTION_INSTR = (
    "You control a simple form UI by emitting ONE action as JSON.\n"
    "Markers: 4 = search box, 7 = Submit button.\n"
    "Goal: type the word 'cats' into the search box, then submit.\n"
    "Valid actions:\n"
    '  {\"type\": \"type\", \"mark\": 4, \"text\": \"cats\"}\n'
    '  {\"type\": \"click\", \"mark\": 7}\n'
    '  {\"type\": \"done\"}\n'
    "Look at the current screenshot and output ONLY the next single action as JSON."
)

def parse_action(text):
    mobj = re.search(r'\{.*\}', text, re.S)
    if not mobj:
        return {"type": "noop"}
    try:
        return _json.loads(mobj.group())
    except Exception:
        return {"type": "noop"}

env = ToyFormEnv()
MAX_STEPS = 3
for step in range(1, MAX_STEPS + 1):
    obs = env.render()                       # 观察：渲染当前状态成图
    raw = ask_qwen(obs, ACTION_INSTR, max_new_tokens=48)   # 推理：VLM 给动作
    action = parse_action(raw)
    print(f"[step {step}] VLM 原始: {raw.strip()[:120]!r}")
    print(f"[step {step}] 解析动作: {action}")
    if action.get("type") == "done":
        print("→ agent 主动结束。"); break
    new_state = env.apply(action)            # 执行：在 toy 环境里 apply
    print(f"[step {step}] 新状态: {new_state}\n")
    if new_state["submitted"]:
        print("✅ 任务完成：已输入并提交。"); break

print("动作日志:", env.action_log)
print("\n注意：2B 小模型可能不会一次走对，这正体现了第 7 节讲的【多步可靠性】难题——"
      "单步小概率出错，长轨迹会累积放大。生产 agent 需自检/重试/护栏。")
env.render()


## 7. (可选) 调用闭源原生多模态 GPT-4o

若环境变量里检测到 `OPENAI_API_KEY`，就用 `openai` SDK 对界面图做一次**原生多模态问答**，
对照体会"后期拼接的开源 VLM" vs "原生多模态闭源模型"的差异。没有 key 则自动跳过。

In [ ]:
import os, base64

if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        buf = io.BytesIO(); ui_img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        client = OpenAI()
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": [
                {"type": "text", "text": "Describe the UI and tell me where the Submit button is."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}],
            max_tokens=200,
        )
        print("GPT-4o 回答：\n", resp.choices[0].message.content)
    except Exception as e:
        print("调用 GPT-4o 失败：", e)
else:
    print("未检测到 OPENAI_API_KEY，跳过闭源调用。")
    print("（设置 export OPENAI_API_KEY=... 后重跑本 cell 即可用 GPT-4o 做原生多模态问答。）")


---
## ✏️ 练习 1：实现 `iou` 与 `click_hit`

GUI grounding 的判分核心就两个几何量：预测框与真值框的 **IoU**（Intersection over Union），以及预测的点击点是否落在目标控件框内（ScreenSpot 等 grounding benchmark 的判分标准）。框用像素坐标 `(x1, y1, x2, y2)` 表示，保证 `x1<=x2, y1<=y2`。

- `iou(box_a, box_b)`：返回交并比 float；不相交返回 `0.0`。
- `click_hit(point, box)`：`point=(x, y)` 落在框内（**含边界**）返回 `True`。

**提示**：交集宽 = `max(0, min(ax2, bx2) - max(ax1, bx1))`，高同理；并集 = 两框面积之和 − 交集；并集为 0（两个零面积框）时返回 `0.0` 防除零。`click_hit` 用链式比较 `x1 <= x <= x2` 一行搞定。两个函数合计 15 行以内。

In [ ]:
def iou(box_a, box_b):
    # TODO: 计算两个 (x1, y1, x2, y2) 框的 IoU；不相交返回 0.0
    raise NotImplementedError

def click_hit(point, box):
    # TODO: 判断 point=(x, y) 是否落在 box 内（含边界），返回 bool
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert iou((0, 0, 10, 10), (0, 0, 10, 10)) == 1.0                # 完全重合
assert iou((0, 0, 10, 10), (20, 20, 30, 30)) == 0.0              # 完全不相交
assert abs(iou((0, 0, 10, 10), (5, 0, 15, 10)) - 1/3) < 1e-9     # 交 50 / 并 150
assert iou((0, 0, 10, 10), (10, 0, 20, 10)) == 0.0               # 只共享一条边：交集面积为 0
assert click_hit((5, 5), (0, 0, 10, 10)) is True
assert click_hit((10, 10), (0, 0, 10, 10)) is True               # 边界含端点
assert click_hit((11, 5), (0, 0, 10, 10)) is False
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 `norm_to_pixel` 与 `pixel_to_norm`

Qwen2-VL 习惯输出 **0–1000 归一化坐标**的 bbox（第 4 节 `parse_boxes` 里处理过）。把这套坐标变换抽成两个干净的函数：

- `norm_to_pixel(box, img_w, img_h)`：把 0–1000 归一化 `(x1, y1, x2, y2)` 映射回像素，**clip 到画布内**，并保证返回 `x1<=x2, y1<=y2`（模型可能把两个角点输出反了）。
- `pixel_to_norm(box, img_w, img_h)`：反向映射回 0–1000（无需 clip / 排序）。

**提示**：`x_pixel = x_norm / 1000 * img_w`；clip 用 `max(0.0, min(img_w, x))`；排序用 `sorted([...])` 解包——与第 4 节 `parse_boxes` 的写法一致。两个函数合计 12 行以内，均返回 tuple。

In [ ]:
def norm_to_pixel(box, img_w, img_h):
    # TODO: 0-1000 归一化 (x1,y1,x2,y2) -> 像素坐标
    #       clip 到 [0, img_w] x [0, img_h]，并保证 x1<=x2, y1<=y2
    raise NotImplementedError

def pixel_to_norm(box, img_w, img_h):
    # TODO: 像素坐标 -> 0-1000 归一化（无需 clip / 排序）
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
def _close(a, b, tol=1e-6):
    return all(abs(x - y) <= tol for x, y in zip(a, b))

assert _close(norm_to_pixel((100, 200, 500, 800), 640, 400), (64, 80, 320, 320))
assert _close(norm_to_pixel((500, 800, 100, 200), 640, 400), (64, 80, 320, 320))  # 角点顺序反了也要纠正
assert _close(norm_to_pixel((-50, 0, 1200, 1000), 640, 400), (0, 0, 640, 400))    # 越界 clip 到画布
assert _close(pixel_to_norm((64, 80, 320, 320), 640, 400), (100, 200, 500, 800))
box = (235, 220, 530, 812)
assert _close(pixel_to_norm(norm_to_pixel(box, 640, 400), 640, 400), box)         # round-trip 无损
print("✅ 练习 2 通过")

## ✏️ 练习 3：实现 `parse_action_ex`

第 6 节的 `parse_action` 很脆：模型输出经常带 markdown 代码块包裹、前后解释性废话，甚至缺字段。写一个更鲁棒的版本——从自由文本里抽出**一个** JSON 动作对象并校验：

1. 用正则抓出文本里的 `{...}` 片段并 `json.loads`；
2. 解析结果必须是 dict 且含 `"type"` 字段，否则视为失败；
3. 任何一步失败都返回 `{"type": "noop"}` —— agent 解析失败时宁可 no-op，也不能瞎执行动作（安全兜底）。

**提示**：`re.search(r'\{.*\}', text, re.S)` + `try/except`；`re.S` 让 `.` 能匹配换行，markdown 代码块里的 JSON 自然就能抓到。10 行以内。

In [ ]:
import re, json

def parse_action_ex(text):
    # TODO: 1) 正则抓出 {...} 片段  2) json.loads  3) 校验是 dict 且含 "type"
    #       任何一步失败都返回 {"type": "noop"}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert parse_action_ex('{"type": "click", "mark": 7}') == {"type": "click", "mark": 7}
fenced = '```json\n{"type": "type", "mark": 4, "text": "cats"}\n```'
assert parse_action_ex(fenced) == {"type": "type", "mark": 4, "text": "cats"}      # markdown 代码块包裹
chatty = 'Sure! The next action is {"type": "click", "mark": 7} — good luck!'
assert parse_action_ex(chatty)["mark"] == 7                                        # 前后有废话
assert parse_action_ex("I will click the Submit button now.") == {"type": "noop"}  # 没有 JSON
assert parse_action_ex('{"type": click}') == {"type": "noop"}                      # 非法 JSON
assert parse_action_ex('{"mark": 7}') == {"type": "noop"}                          # 缺 "type" 字段
print("✅ 练习 3 通过")

---
## 📖 参考答案

In [ ]:
# 练习 1 参考答案（先自己做，再对照）
def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    iw = max(0, min(ax2, bx2) - max(ax1, bx1))
    ih = max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else 0.0

def click_hit(point, box):
    x, y = point
    x1, y1, x2, y2 = box
    return x1 <= x <= x2 and y1 <= y <= y2

In [ ]:
# 练习 2 参考答案（先自己做，再对照）
def norm_to_pixel(box, img_w, img_h):
    x1, y1, x2, y2 = box
    x1, x2 = x1 / 1000 * img_w, x2 / 1000 * img_w
    y1, y2 = y1 / 1000 * img_h, y2 / 1000 * img_h
    x1, x2 = sorted([max(0.0, min(img_w, x1)), max(0.0, min(img_w, x2))])
    y1, y2 = sorted([max(0.0, min(img_h, y1)), max(0.0, min(img_h, y2))])
    return (x1, y1, x2, y2)

def pixel_to_norm(box, img_w, img_h):
    x1, y1, x2, y2 = box
    return (x1 / img_w * 1000, y1 / img_h * 1000,
            x2 / img_w * 1000, y2 / img_h * 1000)

In [ ]:
# 练习 3 参考答案（先自己做，再对照）
import re, json

def parse_action_ex(text):
    mobj = re.search(r'\{.*\}', text, re.S)
    if not mobj:
        return {"type": "noop"}
    try:
        obj = json.loads(mobj.group())
    except Exception:
        return {"type": "noop"}
    if not isinstance(obj, dict) or "type" not in obj:
        return {"type": "noop"}
    return obj

## 8. 小结 + 动手练习 + 结课

**本 notebook 串起来的 agent 基础**：
- **GUI grounding**：把指令映射到屏幕坐标 / bbox（第 3–4 节）——agent 的"眼睛 + 手指"。
- **Set-of-Marks**：叠编号把坐标回归降级为离散选择（第 5 节）——解耦感知与决策，提升可靠性。
- **感知-行动循环**：`render → VLM → action → apply` 的闭环（第 6 节）——agent 的骨架；多步可靠性是落地瓶颈。
- **原生 vs 拼接**：GPT-4o 等原生模型把多模态长在一起（第 7 节）。

**动手练习**：
1. 把 toy 任务改成两步（先在搜索框输 "dogs"，再点 Save 而非 Submit），观察 agent 是否仍可靠完成，体会多步误差累积。
2. 关掉 Set-of-Marks（不叠编号），改回让模型直接输出坐标驱动 toy 环境，对比成功率——量化 SoM 的增益。
3. 把 `MAX_STEPS` 调大并在 `ToyFormEnv` 里加一个"危险动作"（如 `delete`），写一个**安全护栏**：执行不可逆动作前必须人类确认。这正是第 6/7 节强调的 agentic safety。

---

### 🎓 全课总结：10 个模块的一句话脉络

> **像素 → 向量（01 视觉编码器）→ 与语言对齐（02 CLIP/SigLIP）→ 接进 LLM（03 架构 / 04 连接器）→ 教它听话（05 指令微调与对齐）→ 看清高清与视频（06）→ 严谨地评测它（07）→ 找出它的幻觉与不安全（08）→ 让它原生多模态并能行动，再去评测这个 agent（09）。**

一条主线贯穿始终：**能力如何构建，边界与失败如何被评测和归因**——这正是 model / agent evaluation 研究科学家的核心工作。

🎉 **恭喜完成全部 10 个模块！** → [回到课程主页 / 结课](../index.html)

---
## 🎯 真实数据胶囊题：真实图像上的区域 grounding

多模态 agent 要把指令 ground 到图像区域。把真实图像切成网格区域，实现按区域平均亮度最高的简单准则 ground 一个指令到具体区域(返回行列)。

> 本模块新增的**真实数据**练习：用**真实图像**把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, io, urllib.request
import numpy as np
import matplotlib.image as mpimg
CACHE=os.path.expanduser("~/.vlm_data"); os.makedirs(CACHE,exist_ok=True)
def real_image():
    "真实图像 Grace Hopper (来自 matplotlib 示例数据), 返回 (H,W,3) uint8"
    p=os.path.join(CACHE,"grace_hopper.jpg")
    if not os.path.exists(p):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/matplotlib/matplotlib/main/lib/matplotlib/mpl-data/sample_data/grace_hopper.jpg", p)
    return mpimg.imread(p)

img=real_image().astype(float).mean(2)/255   # 灰度
K=4; H=img.shape[0]//K*K; W=img.shape[1]//K*K; img=img[:H,:W]
print(f"真实图像切成 {K}x{K} 区域网格")

**练习**：实现 `ground_brightest(img, K)`：把图切成 K×K 区域，返回**平均亮度最高**区域的 `(行, 列)`。(模拟把“点击最亮的区域”这类指令 ground 到具体网格。)

In [ ]:
def ground_brightest(img, K=4):
    # TODO: 切 K*K 网格，算每块均值，返回最大块的 (row, col)
    raise NotImplementedError


In [ ]:
# 自测
r,c = ground_brightest(img, K)
assert 0<=r<K and 0<=c<K
# 验证：返回的块确实是均值最大的
h,w=img.shape[0]//K, img.shape[1]//K
means=np.array([[img[i*h:(i+1)*h, j*w:(j+1)*w].mean() for j in range(K)] for i in range(K)])
assert (r,c)==tuple(np.unravel_index(means.argmax(), means.shape))
print(f"区域 grounding ✓  最亮区域 = 第{r}行第{c}列")


### 📖 参考答案

In [ ]:
def ground_brightest(img, K=4):
    h,w=img.shape[0]//K, img.shape[1]//K
    means=np.array([[img[i*h:(i+1)*h, j*w:(j+1)*w].mean() for j in range(K)] for i in range(K)])
    return tuple(int(x) for x in np.unravel_index(means.argmax(), means.shape))
print("✓ grounding 把指令落到图像具体区域，是多模态 agent 操作的前提")